# S4 -- MCQ Generation Pipeline (v6 - Single-Slot Generation)

**AI Tutor System - Teacher-Student Knowledge Distillation**

---

### Architecture Overview
```
For each chunk:
  |-- clean_concepts()            # validate / fallback-extract concepts
  |-- assign_slots()              # dynamic slots from concept count
  |-- for each slot (one at a time):
       |-- build_user_prompt()       # 1 slot -> 1 prompt
       |-- generate_slot_batch()     # 1 API call -> 1 MCQ
       |-- save_raw_output()         # Stage 1: raw LLM output
       |-- parse_json_safe()         # parse single object response
       |-- save_parsed_output()      # Stage 2: parsed output
       |-- validate MCQ (Pydantic + semantic)
       |-- shuffle options
       |-- save_slot_result()        # Stage 3: per-slot checkpoint
  |-- combine all valid MCQs
  |-- save raw (all MCQs) to S4_ALL_PATH
  |-- split -> success.jsonl / failed.jsonl
  |-- attach SFT record to success records
```

### Key Design Decisions (v6 changes)
| Decision | Reason |
|---|---|
| 1 slot per API call | Simpler, more reliable, easier to debug |
| Per-slot resume via S4_SLOTS_PATH | Skip only completed slots, not whole chunks |
| No distractor explanations field | Removed from schema, prompt, and shuffle logic |
| Compressed SYSTEM_PROMPT | Shorter but all constraints preserved |
| Stage 1 raw save (before parse) | Recover raw LLM output on parse failure |
| Stage 2 parsed save | Inspect pre-validation output |
| Per-slot checkpoint (save_slot_result) | Resume granularity = individual slot |

### Slot Assignment Rules
| Concepts available | Slots generated | API calls |
|---|---|---|
| 1 concept | 3 slots | 3 calls |
| 2 concepts | 3 slots | 3 calls |
| >= 3 concepts | 4 slots | 4 calls |

**Model**: `gemini-2.5-flash` - **KD**: Option B (no `question_targets`)


## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
!pip install -qU google-genai==0.8.0
!pip install -qU json-repair==0.29.1
!pip install -qU pydantic>=2.0
!pip install -qU tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.4/125.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 0.8.0 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 14.2 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 0.8.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 0.8.0 which is incompatible.
google-ad

## 2. Imports & Configuration

In [ ]:
import json
import os
import re
import time
import random
import hashlib
import json_repair
import requests
import torch
import threading


from os.path import join
from typing import List, Optional, Dict, Any, Tuple, Literal
from collections import Counter
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, field_validator, model_validator
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google import genai
from google.colab import userdata

# -- Paths --------------------------------------------------------------------
data_dir        = "/gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen"
S3_OUTPUT_PATH  = join(data_dir, "selected_subset_v4_new.jsonl")

# Primary output files
S4_ALL_PATH     = join(data_dir, "s4_all_mcqs.jsonl")       # every MCQ (pass + fail)
S4_SUCCESS_PATH = join(data_dir, "s4_success.jsonl")        # validated MCQs only
S4_FAILED_PATH  = join(data_dir, "s4_failed.jsonl")         # invalid slots only
S4_SFT_PATH     = join(data_dir, "s4_sft_dataset.jsonl")    # LLaMA-Factory training

# -- Stage saving paths (v6: multi-stage observability) ----------------------
S4_RAW_PATH     = join(data_dir, "s4_raw_outputs.jsonl")    # Stage 1: raw LLM text
S4_PARSED_PATH  = join(data_dir, "s4_parsed_outputs.jsonl") # Stage 2: parsed JSON
S4_SLOTS_PATH   = join(data_dir, "s4_slot_results.jsonl")   # Stage 3: per-slot status

os.makedirs(data_dir, exist_ok=True)

# -- Gemini model -------------------------------------------------------------
GEMINI_MODEL   = "gemini-2.5-flash-lite"
# GEMINI_MODEL   = "gemini-2.5-flash"

# -- Generation config --------------------------------------------------------
MAX_RETRIES_PER_SLOT = 1      # retries per slot -- not per chunk
MAX_OUTPUT_TOKENS    = 1800    # 1 MCQ per call
CALL_SLEEP           = 0.5    # gentle rate-limit buffer between calls

# ─────────────────────────────────────────────────────────────────────────────
# Multi-key Gemini API Manager
# ─────────────────────────────────────────────────────────────────────────────

# Add all your Gemini API keys here.
# Keys are loaded from Colab secrets; add as many as you have.
# ── API key loading: reads all keys from a .txt file (one key per line) ──────
# Set this path to your keys file.
# Format: plain text, one API key per line, blank lines and #-comments ignored.
GEMINI_KEYS_FILE = join(data_dir, "All Gemini API Keys.txt")


def _load_gemini_keys(path: str) -> List[str]:
    """
    Load Gemini API keys from a plain-text file.

    File format:
        - One API key per line
        - Blank lines are ignored
        - Lines starting with # are treated as comments and ignored
        - Leading/trailing whitespace is stripped from each key

    Args:
        path: absolute path to the .txt keys file

    Returns:
        List of non-empty API key strings
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Gemini keys file not found: {path}\n"
            "Create a plain-text file with one API key per line and update GEMINI_KEYS_FILE."
        )

    keys: List[str] = []
    with open(path, encoding="utf-8") as fh:
        for lineno, raw_line in enumerate(fh, start=1):
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue   # skip blank lines and comments
            keys.append(line)

    return keys


_RAW_KEYS: List[str] = _load_gemini_keys(GEMINI_KEYS_FILE)

if not _RAW_KEYS:
    raise RuntimeError(
        f"No API keys found in {GEMINI_KEYS_FILE}.\n"
        "Make sure the file exists and contains at least one key (one per line)."
    )

print(f"Loaded {len(_RAW_KEYS)} API key(s) from {GEMINI_KEYS_FILE}")


class GeminiKeyManager:
    """
    Thread-safe round-robin API key manager with per-key cooldown.

    Usage:
        key = manager.get_available_key()
        client = genai.Client(api_key=key)

    On rate-limit error:
        manager.report_rate_limit(key)   # puts key on cooldown
    """

    KEY_COOLDOWN_SECONDS: float = 65.0   # how long a key rests after a 429
    WAIT_POLL_INTERVAL:   float = 5.0    # how often to re-check when all keys exhausted
    MAX_WAIT_SECONDS:     float = 600.0  # give up waiting after 10 minutes total

    def __init__(self, api_keys: List[str]) -> None:
        if not api_keys:
            raise ValueError("GeminiKeyManager requires at least one API key.")
        self._keys: List[str]         = list(api_keys)
        self._cooldown_until: Dict[str, float] = {k: 0.0 for k in self._keys}
        self._lock = threading.Lock()
        self._rr_index: int = 0   # round-robin cursor

        # Build masked labels for safe logging (show first 6 + last 4 chars)
        self._masked: Dict[str, str] = {
            k: f"{k[:6]}...{k[-4:]}" if len(k) > 10 else "***"
            for k in self._keys
        }
        print(f"[KeyManager] Initialized with {len(self._keys)} key(s): "
              f"{[self._masked[k] for k in self._keys]}")

    # ── Public API ───────────────────────────────────────────────────────────

    def get_available_key(self) -> str:
        """
        Return the next available (not-on-cooldown) key using round-robin.
        If all keys are on cooldown, blocks until the soonest one recovers.
        Raises RuntimeError if no key becomes available within MAX_WAIT_SECONDS.
        """
        waited_total = 0.0
        while waited_total < self.MAX_WAIT_SECONDS:
            with self._lock:
                key = self._next_available()
                if key is not None:
                    return key
                # All exhausted -- compute shortest wait
                now      = time.time()
                earliest = min(self._cooldown_until[k] for k in self._keys)
                wait_sec = max(0.0, earliest - now) + 1.0  # +1s safety margin

            print(f"[KeyManager] ⏳ All {len(self._keys)} key(s) on cooldown. "
                  f"Waiting {wait_sec:.1f}s for soonest key to recover...")
            time.sleep(min(wait_sec, self.WAIT_POLL_INTERVAL))
            waited_total += min(wait_sec, self.WAIT_POLL_INTERVAL)

        raise RuntimeError(
            f"[KeyManager] No API key became available after {self.MAX_WAIT_SECONDS}s. "
            "Check your quota limits."
        )

    def report_rate_limit(self, key: str) -> None:
        """
        Mark a key as rate-limited. It will be unavailable for KEY_COOLDOWN_SECONDS.
        Call this whenever you receive a 429 / quota / rate-limit error.
        """
        with self._lock:
            until = time.time() + self.KEY_COOLDOWN_SECONDS
            self._cooldown_until[key] = until
        print(f"[KeyManager] 🔴 Key {self._masked.get(key, '???')} on cooldown "
              f"for {self.KEY_COOLDOWN_SECONDS:.0f}s "
              f"(resumes at {time.strftime('%H:%M:%S', time.localtime(until))})")

    def report_suspended_key(self, key: str) -> None:
        with self._lock:
            self._cooldown_until[key] = time.time() + 10**9  # or remove/disable it permanently
        print(f"[KeyManager] 🚫 Key {self._masked.get(key, '???')} is suspended. Switching key...")

    def report_success(self, key: str) -> None:
        """Optional: called on a successful call to clear any partial backoff."""
        # No-op in the current design; kept for future extension.
        pass

    def _next_available(self) -> Optional[str]:
        """
        Use ONE key continuously until it hits cooldown.
        Only switch when the current key becomes rate-limited.
        """

        now = time.time()

        # Current active key
        current_key = self._keys[self._rr_index]

        # If current key is available -> keep using it
        if self._cooldown_until[current_key] <= now:
            return current_key

        # Current key exhausted -> search for next available key
        n = len(self._keys)

        for offset in range(1, n + 1):
            idx = (self._rr_index + offset) % n
            k = self._keys[idx]

            if self._cooldown_until[k] <= now:
                # Switch permanently to this new key
                self._rr_index = idx

                print(
                    f"[KeyManager] 🔄 Switching API key -> "
                    f"{self._masked.get(k, '???')}"
                )

                return k

        # No available keys
        return None

# Instantiate the global key manager
manager = GeminiKeyManager(_RAW_KEYS)


def _make_client(api_key: str) -> "genai.Client":
    """Create a fresh genai.Client for the given key."""
    return genai.Client(api_key=api_key)


print(f"Model        : {GEMINI_MODEL}")
print(f"Input        : {S3_OUTPUT_PATH}")
print(f"Output (all) : {S4_ALL_PATH}")
print(f"Raw stage    : {S4_RAW_PATH}")
print(f"Parsed stage : {S4_PARSED_PATH}")
print(f"Slot results : {S4_SLOTS_PATH}")


Loaded 235 API key(s) from /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/All Gemini API Keys.txt
[KeyManager] Initialized with 235 key(s): ['AIzaSy...KnPM', 'AIzaSy...NCco', 'AIzaSy...6uF0', 'AIzaSy...dntc', 'AIzaSy...nrRI', 'AIzaSy...HUpc', 'AIzaSy...X6uA', 'AIzaSy...BMqA', 'AIzaSy...bWEM', 'AIzaSy...57Uk', 'AIzaSy...-Mvk', 'AIzaSy..._8Ko', 'AIzaSy...GMcI', 'AIzaSy...XLSc', 'AIzaSy...QXYI', 'AIzaSy...1jrI', 'AIzaSy...vJqU', 'AIzaSy...idM8', 'AIzaSy...KmTA', 'AIzaSy...dbU0', 'AIzaSy...AM3M', 'AIzaSy...xq_4', 'AIzaSy...JmcM', 'AIzaSy...kwWk', 'AIzaSy...kgEw', 'AIzaSy...mnGc', 'AIzaSy...12qs', 'AIzaSy...9mKY', 'AIzaSy...-Mvk', 'AIzaSy..._8Ko', 'AIzaSy...GMcI', 'AIzaSy...XLSc', 'AIzaSy...QXYI', 'AIzaSy...1jrI', 'AIzaSy...vJqU', 'AIzaSy...idM8', 'AIzaSy...KmTA', 'AIzaSy...coe8', 'AIzaSy...WaAM', 'AIzaSy...6Wd0', 'AIzaSy...CqJU', 'AIzaSy...mzQI', 'AIzaSy...2Mpg', 'AIzaSy...aBSk', 'AIzaSy...a_Ps', 'AIzaSy...9FfM', 'AIzaSy...gezg', 'AIzaSy...jU4g', 'AIzaSy...RTL0', 'AIzaSy...0uPI', 'AIzaSy.

In [ ]:
# =============================================================================
# VERIFY THAT S4_SLOTS_PATH AND S3_OUTPUT_PATH MATCH
# =============================================================================

slot_ids = set()

with open(S4_SLOTS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rec = json.loads(line)
            slot_ids.add(rec["chunk_id"])

chunk_ids = set()

with open(S3_OUTPUT_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rec = json.loads(line)
            chunk_ids.add(rec["chunk_id"])

print(f"S4_SLOTS_PATH unique chunk_ids : {len(slot_ids)}")
print(f"S3_OUTPUT_PATH unique chunk_ids: {len(chunk_ids)}")

matches = slot_ids & chunk_ids

print(f"Matching chunk_ids             : {len(matches)}")

# Show examples
print("\nSample slot chunk_ids:")
for x in list(slot_ids)[:5]:
    print(" ", x)

print("\nSample S3 chunk_ids:")
for x in list(chunk_ids)[:5]:
    print(" ", x)

S4_SLOTS_PATH unique chunk_ids : 555
S3_OUTPUT_PATH unique chunk_ids: 555
Matching chunk_ids             : 555

Sample slot chunk_ids:
  foundation_models_for_nlp_578
  foundation_models_for_nlp_481
  hands-on-machine-learning_85
  andrew-ng-machine-learning-yearning_94
  hands-on-machine-learning_722

Sample S3 chunk_ids:
  foundation_models_for_nlp_578
  foundation_models_for_nlp_481
  hands-on-machine-learning_85
  andrew-ng-machine-learning-yearning_94
  hands-on-machine-learning_722


## 3. Temperature Strategy per Bloom Level

Per-slot temperature -- each slot has its own bloom level, so temperature is resolved per slot (not per chunk).

| Bloom | Temp | Reasoning |
|---|---|---|
| remember | 0.2 | Factual recall -- low variance needed |
| understand | 0.3 | Interpretation -- slight creativity |
| apply | 0.5 | Scenario design -- balanced |
| analyze | 0.6 | Comparison / reasoning -- more variation |
| evaluate | 0.7 | Judgment -- needs diverse angles |
| create | 0.8 | Design / synthesis -- highest creativity |

In [ ]:
BLOOM_TEMPERATURE: Dict[str, float] = {
    "remember"  : 0.2,    "understand": 0.3,
    "apply"     : 0.5,
    "analyze"   : 0.6,
    "evaluate"  : 0.7,
    "create"    : 0.8,
}

def get_temperature(bloom_level: str) -> float:
    return BLOOM_TEMPERATURE.get(bloom_level.lower(), 0.4)

## 4. Concept Cleaning & Dynamic Slot Assignment

### `clean_concepts()` -- validates concepts from S3, falls back to NLP extraction
### `assign_slots()` -- derives slots dynamically from concept count

**Dynamic slot rule**:
- 1 concept  -> 2 slots with same concept (different Bloom) + 1 inferred = **3 slots**
- 2 concepts -> 2 concept slots + 1 inferred = **3 slots**
- >=3 concepts -> top-3 concept slots + 1 inferred = **4 slots**

In [ ]:
# -- Constants ----------------------------------------------------------------
MIN_CONCEPT_LENGTH  = 3
MIN_CONCEPTS_NEEDED = 2
_BLOOM_ORDER = ["remember", "understand", "apply", "analyze", "evaluate", "create"]


# -- Concept extraction fallback -----------------------------------------------
def _extract_concepts_from_text(text: str, max_concepts: int = 4) -> List[str]:
    """
    Lightweight NLP-free concept extraction using word-bigram frequency.
    Falls back to longest unique non-stopword words.
    """
    clean    = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)
    words    = clean.split()
    candidates: List[str] = []
    for i in range(len(words) - 1):
        w1, w2 = words[i].lower(), words[i + 1].lower()
        if len(w1) >= 4 and len(w2) >= 4:
            candidates.append(f"{w1} {w2}")

    freq = Counter(candidates)
    top  = [c for c, _ in freq.most_common(max_concepts * 2)]

    if not top:
        stopwords = {"this","that","with","from","have","they","been","were",
                     "their","into","also","than","when","what","which","will","about","more"}
        top = sorted(
            {w.lower() for w in words if len(w) >= 5 and w.lower() not in stopwords},
            key=len, reverse=True
        )[:max_concepts]

    return top[:max_concepts]


def clean_concepts(concepts: List[str], chunk_text: str) -> List[str]:
    """
    Validate concept list from S3. Triggers NLP fallback if concepts are
    missing, too short, or fewer than MIN_CONCEPTS_NEEDED.
    """
    cleaned = [c.strip() for c in concepts
               if isinstance(c, str) and len(c.strip()) >= MIN_CONCEPT_LENGTH]

    if len(cleaned) < MIN_CONCEPTS_NEEDED:
        extracted = _extract_concepts_from_text(chunk_text, max_concepts=4)
        seen: set = set()
        merged: List[str] = []
        for c in cleaned + extracted:
            key = c.lower()
            if key not in seen:
                seen.add(key)
                merged.append(c)
        cleaned = merged[:4]

    return cleaned


# -- Dynamic slot assignment ---------------------------------------------------
def _next_bloom(bloom: str, step: int = 1) -> str:
    """Advance Bloom level by `step` tiers, capped at 'create'."""
    idx = _BLOOM_ORDER.index(bloom) if bloom in _BLOOM_ORDER else 2
    return _BLOOM_ORDER[min(len(_BLOOM_ORDER) - 1, idx + step)]


def assign_slots(concepts: List[str], bloom_level: str) -> List[Dict[str, str]]:
    """
    Build MCQ slots dynamically based on available concept count.

    Rules:
      concepts == 1  -> slot(concept, bloom) + slot(concept, bloom+1) + inferred  = 3 slots
      concepts == 2  -> slot(c1, bloom) + slot(c2, bloom+1) + inferred            = 3 slots
      concepts >= 3  -> slot(c1) + slot(c2) + slot(c3) + inferred                 = 4 slots
      concepts >  3  -> use only the first 3 (most relevant from S3)

    Each slot:  {"concept": str, "bloom_level": str, "slot_type": "named"|"inferred"}
    Total MCQs = min(3, len(concepts)) + 1
    """
    bloom = bloom_level.lower() if bloom_level in _BLOOM_ORDER else "understand"
    n     = len(concepts)

    slots: List[Dict[str, str]] = []

    if n == 0:
        # Edge case -- no concepts at all; use 2 inferred slots
        slots = [
            {"concept": "inferred", "bloom_level": bloom,             "slot_type": "inferred"},
            {"concept": "inferred", "bloom_level": _next_bloom(bloom), "slot_type": "inferred"},
        ]
    elif n == 1:
        # Same concept, two different Bloom depths, then inferred
        slots = [
            {"concept": concepts[0], "bloom_level": bloom,                "slot_type": "named"},
            {"concept": concepts[0], "bloom_level": _next_bloom(bloom, 1), "slot_type": "reuse"},
            {"concept": "inferred",  "bloom_level": bloom,                "slot_type": "inferred"},
        ]
    elif n == 2:
        slots = [
            {"concept": concepts[0], "bloom_level": bloom,                "slot_type": "named"},
            {"concept": concepts[1], "bloom_level": _next_bloom(bloom, 1), "slot_type": "named"},
            {"concept": "inferred",  "bloom_level": bloom,                "slot_type": "inferred"},
        ]
    else:
        # 3 or more concepts -- use top 3
        slots = [
            {"concept": concepts[0], "bloom_level": bloom,                "slot_type": "named"},
            {"concept": concepts[1], "bloom_level": _next_bloom(bloom, 1), "slot_type": "named"},
            {"concept": concepts[2], "bloom_level": _next_bloom(bloom, 2), "slot_type": "named"},
            {"concept": "inferred",  "bloom_level": bloom,                "slot_type": "inferred"},
        ]

    return slots

## 4b. Slot Assignment Demo

In [ ]:
# Demo -- verify dynamic slot logic for all concept-count scenarios
for test_concepts, label in [
    ([], "0 concepts"),
    (["spam detection"], "1 concept"),
    (["spam detection", "pattern recognition"], "2 concepts"),
    (["spam detection", "pattern recognition", "algorithm development"], "3 concepts"),
    (["spam detection", "pattern recognition", "algorithm development", "feature extraction"], "4 concepts"),
]:
    slots = assign_slots(test_concepts, "apply")
    print(f"\n{label} -> {len(slots)} slots:")
    for i, s in enumerate(slots, 1):
        tag = f"  [{s['slot_type']}]"
        print(f"  Slot {i}: {s['concept']:<35} bloom={s['bloom_level']}{tag}")


0 concepts -> 2 slots:
  Slot 1: inferred                            bloom=apply  [inferred]
  Slot 2: inferred                            bloom=analyze  [inferred]

1 concept -> 3 slots:
  Slot 1: spam detection                      bloom=apply  [named]
  Slot 2: spam detection                      bloom=analyze  [reuse]
  Slot 3: inferred                            bloom=apply  [inferred]

2 concepts -> 3 slots:
  Slot 1: spam detection                      bloom=apply  [named]
  Slot 2: pattern recognition                 bloom=analyze  [named]
  Slot 3: inferred                            bloom=apply  [inferred]

3 concepts -> 4 slots:
  Slot 1: spam detection                      bloom=apply  [named]
  Slot 2: pattern recognition                 bloom=analyze  [named]
  Slot 3: algorithm development               bloom=evaluate  [named]
  Slot 4: inferred                            bloom=apply  [inferred]

4 concepts -> 4 slots:
  Slot 1: spam detection                      bloom

## 5. Option Shuffling *(eliminates positional bias)*

Gemini tends to put the correct answer in position A.
`shuffle_options()` randomly permutes A/B/C/D and remaps the answer key
to match the new positions.


In [ ]:
_LABELS = ("A", "B", "C", "D")


def shuffle_options(mcq: Dict[str, Any]) -> Dict[str, Any]:
    """
    Randomly permute option texts across A/B/C/D labels.
    Remaps answer key to new positions.
    Returns a new dict -- does NOT mutate the input.
    """
    mcq     = dict(mcq)
    options = mcq.get("options", {})
    answer  = mcq.get("answer", "")

    if not isinstance(options, dict) or len(options) != 4:
        return mcq

    correct_text = options.get(answer, "")
    texts        = list(options.values())
    random.shuffle(texts)
    new_options  = dict(zip(_LABELS, texts))

    new_answer = next(
        (lbl for lbl, txt in new_options.items() if txt == correct_text),
        answer
    )

    mcq["options"] = new_options
    mcq["answer"]  = new_answer
    return mcq


## 6. System Prompt *(static — sent once, fixed across all calls)*

Contains everything that does **not** change between slots:
- Persona & core behavior rules
- `BLOOM_INSTRUCTION` definitions (all 6 levels)
- `CHUNK_TYPE_FORM` definitions (all 5 types)
- Strict rules: grounding, distractor quality, question phrasing, answer position
- Output JSON format schema

> **Token saving**: sent as `system_instruction` once per `GenerateContentConfig`
> object and NOT repeated in the user turn.


In [ ]:
SYSTEM_PROMPT = "\n".join([
    # -- Persona & core behavior
    "You are an expert educational content designer for Bloom's Taxonomy-aligned assessment.",
    "Generate multiple-choice questions from educational text.",
    "Return ONLY valid JSON. No markdown fences, preambles, or commentary.",
    "",

    # -- Bloom levels (compressed)
    "## BLOOM LEVELS",
    "remember: recall facts | understand: explain meaning | apply: use in scenario",
    "analyze: compare/reason | evaluate: justify decision | create: design/propose",
    "",

    # -- Chunk type forms (compressed)
    "## CHUNK TYPE FORMS",
    "definition: explain concept | process: steps/sequence | comparison: differences",
    "example: real case | argument: evaluate claim",
    "",

    # -- Grounding rule (highest priority)
    "## GROUNDING (highest priority)",
    "- Correct answer MUST be explicitly stated in or directly inferable from the chunk.",
    "- Do NOT use external knowledge.",
    "",

    # -- MCQ quality rules
    "## MCQ RULES",
    "- Exactly 4 options: A, B, C, D. Exactly ONE correct answer.",
    "- Forbidden: 'All of the above', 'None of the above', 'Both A and B'.",
    "- Options must be similar in length (±25-30%) and share the same grammatical structure.",
    "- Options must be mutually exclusive. Do NOT embed the correct answer text in the question.",
    "- Do NOT embed the correct answer text in the question",
    "",

    # -- Distractor rules
    "## DISTRACTORS",
    "- Use the keywords list to craft plausible distractors.",
    "- All 3 wrong options must belong to the same category/domain as the correct answer.",
    "- Distractors must be plausible to a student who skipped the chunk.",
    "",

    # -- Question phrasing
    "## QUESTION PHRASING",
    '- Do NOT say "according to the text", "based on the chunk", or "as stated above".',
    "- Question stem must be self-contained and <= 50 words.",
    "",

    "- You MUST complete the JSON fully. Do not stop mid-output.",
    "- Ensure all 4 options A,B,C,D are present before finishing.",
    "",

    "## EXPLANATION RULES",
    "- Do NOT mention option letters (A, B, C, D).",
    "- Refer to the answer by its content, not its position.",
    "- Do NOT use phrases like 'according to the text' or 'according to the chunk'.",
    "- Justify the correct concept clearly using its meaning.",
    "",

    # -- Output format (single MCQ)
    "## OUTPUT FORMAT -- return ONLY this JSON object:",
    "{",
    '  "question": "<full question stem>",',
    '  "options": {"A": "...", "B": "...", "C": "...", "D": "..."},',
    '  "answer": "<A|B|C|D>",',
    '  "explanation": "<1-2 sentences: why correct, referencing chunk>",',
    '  "concept": "<the concept this question tests>",',
    '  "bloom_level": "<bloom level>"',
    "}",
])

print(f"SYSTEM_PROMPT length : {len(SYSTEM_PROMPT)} chars")
print(f"Approx token estimate: ~{len(SYSTEM_PROMPT)//4} tokens")
# SYSTEM_PROMPT length : 1727 chars
# Approx token estimate: ~431 tokens


SYSTEM_PROMPT length : 2164 chars
Approx token estimate: ~541 tokens


### 7. User Prompt Builder *(dynamic — rebuilt for every slot)*

`build_user_prompt()` — single-slot version used for every API call.

`_build_concept_block()` handles the concept-constraint paragraph.

Global rules, BLOOM_INSTRUCTION dict, CHUNK_TYPE_FORM dict, and output schema
are **not repeated here** — they live in SYSTEM_PROMPT.


In [ ]:
# These are kept as Python dicts for the user-prompt lookup
# (so we can inject only the relevant instruction into the user turn).
# The full text of every entry is also in SYSTEM_PROMPT (static reference).

BLOOM_INSTRUCTION: Dict[str, str] = {
    "remember"  : "Generate a RECALL question. Test a fact, definition, or named item directly stated in the text.",
    "understand": "Generate an INTERPRETATION question. Test whether the student can explain or paraphrase.",
    "apply"     : "Generate a SCENARIO question. Place the concept in a new, concrete situation.",
    "analyze"   : "Generate a COMPARISON or REASONING question. The stem MUST compare two items or ask for a cause.",
    "evaluate"  : "Generate a JUDGMENT question. Ask the student to choose and justify.",
    "create"    : "Generate a DESIGN or SYNTHESIS question.",
}

CHUNK_TYPE_FORM: Dict[str, str] = {
    "definition":  "Use 'What is / Which best describes' style.",
    "process":     "Use 'What happens when / Which step follows' style.",
    "comparison":  "Use 'How does X differ from Y / Which distinguishes' style.",
    "example":     "Use a scenario-based style embedding the concept in a concrete case.",
    "argument":    "Require evaluating a claim, premise, or conclusion.",
}


def _build_concept_block(slot: Dict[str, str], all_concepts: List[str]) -> str:
    """Return the concept-constraint paragraph for a single slot."""
    slot_concept = slot["concept"]
    slot_type    = slot.get("slot_type", "named")

    if slot_type == "inferred":
        already_used = ", ".join(f'"{c}"' for c in all_concepts) or "(none)"
        return (
            "CONCEPT CONSTRAINT -- INFERRED\n"
            "The question must test a NEW idea found in the chunk that is NOT any of these:\n"
            f"  Already covered: {already_used}\n"
            "Identify a distinct, meaningful concept from the text and write your question about it.\n"
            'Set the "concept" field to the NAME of the new concept you chose (NOT the word "inferred").'
        )
    elif slot_type == "reuse":
        return (
            "CONCEPT CONSTRAINT -- SAME CONCEPT, DIFFERENT ANGLE\n"
            f'Concept to test: "{slot_concept}"\n'
            "This concept was already tested at a lower Bloom level.\n"
            "You MUST test a DIFFERENT ASPECT or IMPLICATION of this concept.\n"
            "Do NOT ask the same type of question"
        )
    else:
        return (
            "CONCEPT CONSTRAINT\n"
            f'The question MUST focus specifically on: "{slot_concept}"\n'
            "Do NOT drift into other concepts. The entire question, correct answer, and distractors\n"
            f'must be directly connected to "{slot_concept}" as discussed in the chunk.'
        )


def build_user_prompt(
    chunk:        Dict[str, Any],
    slot:         Dict[str, str],
    all_concepts: List[str],
) -> str:
    """
    Build the USER turn for generating EXACTLY ONE MCQ for a given slot.
    Returns a prompt that instructs the model to output ONE JSON object.
    """
    text        = chunk["text"].strip()
    meta        = chunk.get("metadata", {})
    fringe      = chunk.get("context_fringe", {})
    chunk_type  = meta.get("chunk_type", "definition").lower()
    keywords    = meta.get("keywords", [])

    slot_bloom   = slot["bloom_level"]
    prev_s = fringe.get("prev_sentence", "") or "N/A"
    next_s = fringe.get("next_sentence", "") or "N/A"

    bloom_instr = BLOOM_INSTRUCTION.get(slot_bloom, BLOOM_INSTRUCTION["understand"])
    ct_instr    = CHUNK_TYPE_FORM.get(chunk_type, "Use a general question style appropriate to the content.")
    kw_str      = ", ".join(keywords) if keywords else "(infer from text)"
    concept_block = _build_concept_block(slot, all_concepts)

    return "\n".join([
        "## CHUNK TEXT",
        text,
        "",
        "---",
        "",
        "## CONTEXT (background only -- do NOT write questions about these sentences)",
        f"Previous: {prev_s}",
        f"Next    : {next_s}",
        "",
        "---",
        "",
        "## METADATA",
        f"- chunk_type : {chunk_type}",
        f"- keywords   : {kw_str}",
        "",
        "---",
        "",
        "## YOUR TASK FOR THIS QUESTION",
        "Generate EXACTLY ONE MCQ. Not two. Not zero. One.",
        "Return a single JSON OBJECT (not an array).",
        "",
        concept_block,
        "",
        f"BLOOM LEVEL: {slot_bloom.upper()}",
        bloom_instr,
        "",
        "CHUNK TYPE INSTRUCTION",
        ct_instr,
    ])

In [ ]:
# =============================================================================
# v5 STAGE-SAVING HELPERS
# Three levels of saving for full observability and debuggability.
# All saves are append-only ("a" mode) so they survive partial runs.
# =============================================================================

def save_raw_output(record: Dict[str, Any]) -> None:
    """
    Stage 1 -- Save raw LLM text IMMEDIATELY after response.text is read.
    Happens BEFORE any parsing attempt.

    record schema:
        chunk_id    : str   -- chunk this batch belongs to
        batch_index : int   -- 0-based batch index within the chunk
        slot_indices: list  -- list of 0-based slot indices in this batch
        raw_output  : str   -- raw text returned by the LLM
        attempt     : int   -- 1-based retry attempt number
        timestamp   : float -- unix timestamp
    """
    record["timestamp"] = time.time()
    with open(S4_RAW_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_parsed_output(record: Dict[str, Any]) -> None:
    """
    Stage 2 -- Save parsed JSON output AFTER successful parse, BEFORE validation.
    Only called when parse_json_safe / parse_json_safe_list returns non-None.

    record schema:
        chunk_id      : str   -- chunk this batch belongs to
        batch_index   : int   -- 0-based batch index within the chunk
        slot_indices  : list  -- list of 0-based slot indices in this batch
        parsed_output : any   -- parsed Python object (dict or list)
        attempt       : int   -- 1-based retry attempt number
        timestamp     : float -- unix timestamp
    """
    record["timestamp"] = time.time()
    with open(S4_PARSED_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_slot_result(record: Dict[str, Any]) -> None:
    """
    Stage 3 -- Per-slot checkpoint AFTER validation + shuffle (or after failure).
    One record per slot. Written immediately when a slot resolves.

    record schema:
        chunk_id    : str        -- chunk this slot belongs to
        slot_index  : int        -- 0-based slot index
        batch_index : int        -- 0-based batch index within the chunk
        status      : str        -- "success" | "failed"
        mcq         : dict|None  -- validated+shuffled MCQ dict on success
        reason      : str        -- "OK" on success, failure reason otherwise
        timestamp   : float      -- unix timestamp
    """
    record["timestamp"] = time.time()
    with open(S4_SLOTS_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


print("Stage-saving helpers registered: save_raw_output, save_parsed_output, save_slot_result")


Stage-saving helpers registered: save_raw_output, save_parsed_output, save_slot_result


## 8. Validation *(Pydantic schema + semantic checks)*

Two-layer validation:

**Layer 1 -- Pydantic** (`MCQSchema`): Catches wrong types, missing fields,
invalid option labels, invalid answer key -- before any semantic logic runs.

Validation is **per-slot** (one MCQ), not per-chunk.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator, model_validator

# -- Pydantic schema -----------------------------------------------------------
BloomLevel   = Literal["remember", "understand", "apply", "analyze", "evaluate", "create"]
AnswerLabel  = Literal["A", "B", "C", "D"]

class MCQSchema(BaseModel):
    """
    Strict Pydantic v2 schema for a single generated MCQ.
    Pydantic raises ValidationError on any structural violation --
    we catch it in the generator and count it as a retryable failure.
    """
    question:    str = Field(..., min_length=10, max_length=400)
    options:     Dict[AnswerLabel, str]
    answer:      AnswerLabel
    explanation: str = Field(..., min_length=10)
    concept:     str = Field(..., min_length=2)
    bloom_level: BloomLevel

    @field_validator("options")
    @classmethod
    def options_must_have_abcd(cls, v: dict) -> dict:
        if set(v.keys()) != {"A", "B", "C", "D"}:
            raise ValueError(f"options keys must be exactly A,B,C,D -- got {sorted(v.keys())}")
        for lbl, text in v.items():
            if not text or not text.strip():
                raise ValueError(f"option {lbl} is empty")
        return v

    @field_validator("question")
    @classmethod
    def question_word_limit(cls, v: str) -> str:
        if len(v.split()) > 60:
            raise ValueError(f"question too long: {len(v.split())} words")
        return v.strip()

    @model_validator(mode="after")
    def answer_must_be_in_options(self) -> "MCQSchema":
        if self.answer not in self.options:
            raise ValueError(f"answer '{self.answer}' not found in options dict")
        return self

    @model_validator(mode="after")
    def no_banned_option_phrases(self) -> "MCQSchema":
        banned = ["all of the above", "none of the above", "both a and", "both b and"]
        for lbl, text in self.options.items():
            if any(b in text.lower() for b in banned):
                raise ValueError(f"option {lbl} contains banned phrase")
        return self


In [ ]:
# -- Semantic constants --------------------------------------------------------
MIN_EXPLANATION_CHARS           = 30
MIN_DISTRACTOR_CHARS            = 4
MIN_ANSWER_WORDS_IN_CHUNK_RATIO = 0.25

# -- Cross-chunk dedup store (hash-based, no embeddings) ----------------------
global_question_bank: set = set()


def _q_fingerprint(question: str) -> str:
    """MD5 of normalised question string -- for O(1) duplicate lookup."""
    norm = re.sub(r"\W+", "", question.lower())
    return hashlib.md5(norm.encode()).hexdigest()

## 9. Single-Slot Generator *(v6)*

Three functions:
- `parse_json_safe_list()` -- parses a JSON array (or single object -> wraps in list)
- `generate_slot_batch()` -- **one API call per slot**, with per-slot retry
- `generate_all_slots()` -- orchestrates all slots for a chunk, one at a time

**Single-slot logic**: each slot generates exactly one MCQ per API call using
`build_user_prompt()`. The `batch` parameter always contains exactly one slot.

**Stage saves inside `generate_slot_batch()`**:
- Stage 1 (`save_raw_output`) — immediately after `response.text`
- Stage 2 (`save_parsed_output`) — immediately after successful parse
- Stage 3 (`save_slot_result`) — after the slot resolves (success or failure)

**Retry is per slot** — if a slot fails, only that slot is retried.


In [ ]:
def parse_json_safe_list(text: str) -> Optional[List[Dict]]:
    """
    Parse LLM response expected to be a JSON array of MCQ objects.
    Handles:
      - Proper JSON arrays          -> returned as-is (list)
      - Single JSON object          -> wrapped in a list of length 1
      - Markdown fence stripping
      - json_repair fallback

    Returns None if parsing fails entirely.
    """
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text  = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    text = text.strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = json_repair.loads(text)
        except Exception:
            return None

    if isinstance(parsed, list):
        return parsed
    if isinstance(parsed, dict):
        return [parsed]
    return None


def parse_json_safe(text: str) -> Optional[Dict]:
    """
    JSON parse for a single MCQ object.
    Handles markdown fences and json_repair fallback.
    """
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text  = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    try:
        return json_repair.loads(text)
    except Exception:
        return None


def _validate_and_finalize_mcq(
    parsed_mcq:  Dict,
    slot:        Dict[str, str],
    slot_idx:    int,
    chunk_text:  str,
    attempt:     int,
    label:       str,
) -> Tuple[Optional[Dict], str]:
    """
    Shared validation + shuffle pipeline for one parsed MCQ dict.
    Used by both the main generator and the retry path.

    Returns (mcq_dict, "OK") on success, (None, reason) on failure.
    """
    # -- Layer 1: Pydantic structural validation ------------------------------
    try:
        validated = MCQSchema.model_validate(parsed_mcq)
    except Exception as pydantic_err:
        reason = f"Pydantic: {str(pydantic_err)[:120]}"
        print(f"    [{label} | Attempt {attempt}] {reason}")
        return None, reason

    mcq_dict = validated.model_dump()

    # -- Shuffle options to eliminate positional bias -------------------------
    mcq_dict = shuffle_options(mcq_dict)

    # -- Register in cross-chunk dedup bank -----------------------------------
    global_question_bank.add(_q_fingerprint(mcq_dict.get("question", "")))

    # Attach slot metadata for traceability
    mcq_dict["slot_index"] = slot_idx
    mcq_dict["slot_type"]  = slot.get("slot_type", "named")

    return mcq_dict, "OK"


# ── Exponential back-off constants (applied on top of key rotation) ──────────
_BACKOFF_BASE      = 2.0   # seconds for attempt 1
_BACKOFF_CAP       = 60.0  # maximum single wait
_TRANSIENT_MARKERS = ("429", "quota", "rate", "resource exhausted",
                       "service unavailable", "503", "500", "timeout")


def _is_transient(err_str: str) -> bool:
    """Return True if the error string looks like a retryable / transient failure."""
    low = err_str.lower()
    return any(m in low for m in _TRANSIENT_MARKERS)


def generate_slot_batch(
    chunk:        Dict[str, Any],
    batch:        List[Dict[str, str]],
    all_concepts: List[str],
    batch_index:  int,
    slot_indices: List[int],
) -> List[Tuple[Optional[Dict], str]]:
    """
    Generate EXACTLY ONE MCQ in a single API call (single-slot generation).

    API key rotation: each attempt requests a fresh key from `manager`.
    On 429/quota errors the offending key is cooled down and a different
    key is used on the very next attempt.  If all keys are exhausted the
    manager blocks until one recovers.

    Args:
        chunk:        S3 chunk dict
        batch:        list of exactly 1 slot dict
        all_concepts: full named concept list (for inferred-slot instruction)
        batch_index:  0-based slot index within the chunk (for logging)
        slot_indices: list of exactly 1 slot index

    Returns:
        List of (mcq_dict|None, reason_str) -- always length 1.
    """
    chunk_id   = chunk.get("chunk_id", "")
    chunk_text = chunk.get("text", "")
    slot       = batch[0]
    slot_idx   = slot_indices[0]
    label      = f"Slot {slot_idx + 1}"

    temperature = get_temperature(slot["bloom_level"])
    user_prompt = build_user_prompt(chunk, slot, all_concepts)

    # Result accumulator (None = not yet resolved)
    slot_result: Optional[Tuple[Optional[Dict], str]] = None

    for attempt in range(1, MAX_RETRIES_PER_SLOT + 2):
        # +2 so that MAX_RETRIES_PER_SLOT=1 gives attempt 1 (normal) + attempt 2 (1 retry)

        if attempt > 1:
            # Exponential backoff for non-rate-limit retries (rate-limit path
            # is handled separately below via key rotation).
            backoff = min(_BACKOFF_CAP, _BACKOFF_BASE * (2 ** (attempt - 2)))
            print(f"    [{label} | Attempt {attempt}] Backoff {backoff:.1f}s before retry...")
            time.sleep(backoff)

        # ── Key rotation: get a live key for this attempt ────────────────────
        current_key = manager.get_available_key()
        gemini_client_local = _make_client(current_key)
        masked_key = current_key[:6] + "..." + current_key[-4:]
        print(f"    [{label} | Attempt {attempt}] Using key {masked_key}")

        try:
            response = gemini_client_local.models.generate_content(
                model=GEMINI_MODEL,
                contents=user_prompt,
                config=genai.types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=temperature,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                )
            )

            raw = response.text
            manager.report_success(current_key)

            # ── Stage 1: save raw output BEFORE any parsing ──────────────────
            save_raw_output({
                "chunk_id"    : chunk_id,
                "batch_index" : batch_index,
                "slot_indices": slot_indices,
                "raw_output"  : raw,
                "attempt"     : attempt,
            })

            # -- Parse response as single object ------------------------------
            parsed_obj  = parse_json_safe(raw)
            parsed_list = [parsed_obj] if parsed_obj is not None else None

            if parsed_list is None:
                print(f"    [{label} | Attempt {attempt}] JSON parse failed")
                continue

            # ── Stage 2: save parsed output AFTER parse, BEFORE validation ───
            save_parsed_output({
                "chunk_id"     : chunk_id,
                "batch_index"  : batch_index,
                "slot_indices" : slot_indices,
                "parsed_output": parsed_list,
                "attempt"      : attempt,
            })

            # -- Validate the single MCQ --------------------------------------
            mcq_dict, reason = _validate_and_finalize_mcq(
                parsed_list[0], slot, slot_idx, chunk_text, attempt, label
            )

            if mcq_dict is not None:
                slot_result = (mcq_dict, "OK")
                # ── Stage 3: per-slot success checkpoint ─────────────────────
                save_slot_result({
                    "chunk_id"   : chunk_id,
                    "slot_index" : slot_idx,
                    "batch_index": batch_index,
                    "status"     : "success",
                    "mcq"        : mcq_dict,
                    "reason"     : "OK",
                })
                break   # success -- no more retries needed

        except Exception as e:
            err_str = str(e)

            # Suspended / invalid key
            if "PERMISSION_DENIED" in err_str or "has been suspended" in err_str:
                manager.report_suspended_key(current_key)

                print(f"    [{label} | Attempt {attempt}] "
                      f"Suspended key {masked_key}: {err_str[:120]}")

                print(f"    [{label} | Attempt {attempt}] "
                      f"Switching to another API key.")

                continue

            # Transient errors (429 / quota / timeout ...)
            elif _is_transient(err_str):

                manager.report_rate_limit(current_key)

                print(f"    [{label} | Attempt {attempt}] "
                      f"Transient error on key {masked_key}: {err_str[:120]}")

                print(f"    [{label} | Attempt {attempt}] "
                      f"Key rotated -- next attempt will use a different key.")

                continue

            # Other non-transient errors
            else:
                print(f"    [{label} | Attempt {attempt}] "
                      f"Non-transient API error: {err_str[:200]}")

    # -- Finalise: fill unresolved slot with failure result ------------------
    if slot_result is None:
        failure_reason = f"Exhausted {MAX_RETRIES_PER_SLOT} retries"
        slot_result = (None, failure_reason)
        # ── Stage 3: per-slot failure checkpoint ─────────────────────────────
        save_slot_result({
            "chunk_id"   : chunk_id,
            "slot_index" : slot_idx,
            "batch_index": batch_index,
            "status"     : "failed",
            "mcq"        : None,
            "reason"     : failure_reason,
        })

    return [slot_result]


def generate_all_slots(
    chunk:           Dict[str, Any],
    completed_slots: set,
) -> Dict[str, Any]:
    """
    Orchestrate generation for all slots of a chunk, one slot per API call.

    Skips slots whose (chunk_id, slot_index) pair is already in completed_slots.
    A chunk is only fully skipped when ALL its slots are in completed_slots.

    Args:
        chunk:           S3 chunk dict
        completed_slots: set of (chunk_id, slot_index) tuples already done

    Returns a dict with:
        chunk_id:      str
        bloom_level:   str
        chunk_type:    str
        mcqs:          List[Dict]  -- all valid MCQs (successful slots)
        failed_slots:  List[Dict]  -- info about failed slots
        total_slots:   int
        success_count: int
        failure_count: int
    """
    meta        = chunk.get("metadata", {})
    chunk_id    = chunk.get("chunk_id", "")
    bloom_level = meta.get("bloom_level", "understand")
    chunk_type  = meta.get("chunk_type", "definition")
    chunk_text  = chunk.get("text", "")

    # Step 1: clean concepts
    raw_concepts = meta.get("concepts", [])
    concepts     = clean_concepts(raw_concepts, chunk_text)

    # Step 2: assign slots
    slots = assign_slots(concepts, bloom_level)

    valid_mcqs:   List[Dict] = []
    failed_slots: List[Dict] = []

    # Step 3: process each slot individually -- one API call per slot
    for slot_index, slot in enumerate(slots):

        # Per-slot resume: skip if this (chunk_id, slot_index) was already completed
        if (chunk_id, slot_index) in completed_slots:
            print(f"  Slot {slot_index + 1}/{len(slots)} [SKIP -- already done]: "
                  f"concept='{slot['concept']}' bloom={slot['bloom_level']}")
            continue

        print(f"  Slot {slot_index + 1}/{len(slots)}: "
              f"concept='{slot['concept']}' bloom={slot['bloom_level']}")

        results = generate_slot_batch(
            chunk        = chunk,
            batch        = [slot],
            all_concepts = concepts,
            batch_index  = slot_index,
            slot_indices = [slot_index],
        )
        time.sleep(CALL_SLEEP)

        mcq, reason = results[0]
        if mcq is not None:
            valid_mcqs.append(mcq)
        else:
            failed_slots.append({
                "slot_index":  slot_index,
                "concept":     slot["concept"],
                "bloom_level": slot["bloom_level"],
                "slot_type":   slot.get("slot_type", "named"),
                "reason":      reason,
            })

    return {
        "chunk_id":      chunk_id,
        "bloom_level":   bloom_level,
        "chunk_type":    chunk_type,
        "mcqs":          valid_mcqs,
        "failed_slots":  failed_slots,
        "total_slots":   len(slots),
        "success_count": len(valid_mcqs),
        "failure_count": len(failed_slots),
    }


## 10. SFT Record Builder *(LLaMA-Factory format)*

Builds the training record in the `{system, instruction, input, output, history}`
format from the reference notebook.

**Option B KD alignment**: instruction = chunk text + metadata only.
No `question_targets`. Train input == Inference input.

In [ ]:
SFT_SYSTEM = "\n".join([
    "You are an expert AI educator.",
    "Given a chunk of text from an AI textbook and its metadata,",
    "generate a set of high-quality MCQs testing different concepts.",
    "Each MCQ must be grounded in the chunk text.",
    "Return ONLY valid JSON -- no markdown, no preamble.",
])


def build_sft_record(chunk: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build a LLaMA-Factory SFT record from a successful chunk result.

    Follows the reference notebook structure:
        {"system": ..., "instruction": ..., "input": "", "output": ..., "history": []}

    instruction = chunk text + metadata (Option B -- no question_targets)
    output      = JSON array of validated MCQs
    """
    meta   = chunk.get("metadata", {})
    fringe = chunk.get("context_fringe", {})

    ctx_lines = []
    if fringe.get("prev_sentence"):
        ctx_lines.append(f"[Before]: {fringe['prev_sentence']}")
    if fringe.get("next_sentence"):
        ctx_lines.append(f"[After]:  {fringe['next_sentence']}")

    instruction = "\n".join(filter(None, [
        "## Chunk Text:",
        chunk.get("text", "").strip(),
        ("\n## Context:\n" + "\n".join(ctx_lines)) if ctx_lines else "",
        "",
        "## Metadata:",
        f"  bloom_level : {meta.get('bloom_level', 'understand')}",
        f"  chunk_type  : {meta.get('chunk_type', 'definition')}",
        f"  concepts    : {', '.join(meta.get('concepts', []))}",
        f"  keywords    : {', '.join(meta.get('keywords', []))}",
        "",
        "## Output JSON:",
        "```json",
    ]))

    output = "\n".join([
        "```json",
        json.dumps({"mcqs": result.get("mcqs", [])}, ensure_ascii=False, indent=2),
        "```",
    ])

    return {
        "system":      SFT_SYSTEM,
        "instruction": instruction,
        "input":       "",
        "output":      output,
        "history":     [],
        # provenance -- not used by LLaMA-Factory but useful for debugging
        "chunk_id":    result.get("chunk_id"),
        "bloom_level": result.get("bloom_level"),
    }

## 11. Full Pipeline

**Three-phase loop**:
1. **Generate**: run all slots for every chunk -> append to `S4_ALL_PATH` (raw)
2. **Split**: separate valid MCQs -> `S4_SUCCESS_PATH`, failed slots -> `S4_FAILED_PATH`
3. **SFT**: attach training records to success records -> `S4_SFT_PATH`

`S4_ALL_PATH` is written first -- if the notebook crashes mid-run, no MCQs are lost.

In [ ]:
# -- Load S3 chunks -----------------------------------------------------------
s3_chunks: List[Dict] = []
with open(S3_OUTPUT_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            s3_chunks.append(json.loads(line))

print(f"Loaded {len(s3_chunks)} S3 chunks.")

# -- Resume: load completed (chunk_id, slot_index) pairs from S4_SLOTS_PATH --
# Only 'success' records count as completed. Failed slots will be retried.
completed_slots: set = set()   # set of (chunk_id, slot_index) already done
if os.path.exists(S4_SLOTS_PATH):
    with open(S4_SLOTS_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                if rec.get("status") == "success":

                # if rec.get("status") in ["success", "failed"]:
                    completed_slots.add((rec.get("chunk_id", ""), rec.get("slot_index", -1)))

print(f"Already completed: {len(completed_slots)} slots -- will skip individually.")


Loaded 555 S3 chunks.
Already completed: 2200 slots -- will skip individually.


In [ ]:
# -- Phase 1: Generation Loop ------------------------------------------------
success_count  = 0
skip_count     = 0   # chunks where ALL slots were already done
partial_count  = 0   # chunks with >=1 valid MCQ but some failed slots

for chunk in tqdm(s3_chunks, desc="Generating MCQs"):
    chunk_id = chunk.get("chunk_id", "")

    print(f"\nChunk: {chunk_id}")
    result = generate_all_slots(chunk, completed_slots)

    # Write to S4_ALL_PATH only if any slot was processed this run
    # if result["success_count"] > 0 or result["failure_count"] > 0:
    with open(S4_ALL_PATH, "a", encoding="utf-8") as dest:
        dest.write(json.dumps(result, ensure_ascii=False) + "\n")

    if result["success_count"] > 0:
        success_count += 1
        if result["failure_count"] > 0:
            partial_count += 1
    elif result["success_count"] == 0 and result["failure_count"] == 0:
        skip_count += 1   # all slots for this chunk were already completed

print(f"\nChunks with >=1 new MCQ : {success_count}")
print(f"Partially failed         : {partial_count}")
print(f"Fully skipped (all done) : {skip_count}")
print(f"Raw output               : {S4_ALL_PATH}")
print(f"Stage 1 (raw LLM)        : {S4_RAW_PATH}")
print(f"Stage 2 (parsed JSON)    : {S4_PARSED_PATH}")
print(f"Stage 3 (slot results)   : {S4_SLOTS_PATH}")

Generating MCQs:   0%|          | 0/555 [00:00<?, ?it/s]


Chunk: hands-on-machine-learning_672
  Slot 1/4 [SKIP -- already done]: concept='numerical stability' bloom=understand
  Slot 2/4 [SKIP -- already done]: concept='softplus function' bloom=apply
  Slot 3/4 [SKIP -- already done]: concept='exponential term' bloom=analyze
  Slot 4/4 [SKIP -- already done]: concept='inferred' bloom=understand

Chunk: hands-on-machine-learning_1250
  Slot 1/4 [SKIP -- already done]: concept='proximal policy optimization' bloom=analyze
  Slot 2/4 [SKIP -- already done]: concept='sample efficiency' bloom=evaluate
  Slot 3/4 [SKIP -- already done]: concept='trust region policy optimization' bloom=create
  Slot 4/4 [SKIP -- already done]: concept='inferred' bloom=analyze

Chunk: andrew-ng-machine-learning-yearning_103
  Slot 1/4 [SKIP -- already done]: concept='data consistency' bloom=understand
  Slot 2/4 [SKIP -- already done]: concept='task reliability' bloom=apply
  Slot 3/4 [SKIP -- already done]: concept='function mapping' bloom=analyze
  Slot 4/4 [SKIP 

In [ ]:
# -- Phase 2: Split into success / failed -------------------------------------
all_results: List[Dict] = []
with open(S4_ALL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            all_results.append(json.loads(line))


def split_results(
    results:       List[Dict],
    success_path:  str,
    failed_path:   str,
    sft_path:      str,
    chunks_lookup: Dict[str, Dict],
) -> Tuple[int, int, int]:
    """
    Write success records (chunks with >=1 valid MCQ) to success_path.
    Write all failed_slots to failed_path.
    Write SFT records for success chunks to sft_path.

    Returns (n_success_chunks, n_failed_slot_records, n_sft_records).
    """
    n_success = n_failed = n_sft = 0

    with open(success_path, "w", encoding="utf-8") as sf,          open(failed_path,  "w", encoding="utf-8") as ff,          open(sft_path,     "w", encoding="utf-8") as sftf:

        for result in results:
            chunk_id = result.get("chunk_id", "")

            # -- Success records -----------------------------------------------
            if result.get("success_count", 0) > 0:
                sf.write(json.dumps(result, ensure_ascii=False) + "\n")
                n_success += 1

                # -- SFT record ------------------------------------------------
                source_chunk = chunks_lookup.get(chunk_id)
                if source_chunk:
                    sft_rec = build_sft_record(source_chunk, result)
                    sftf.write(json.dumps(sft_rec, ensure_ascii=False) + "\n")
                    n_sft += 1

            # -- Failed slot records -------------------------------------------
            for failed_slot in result.get("failed_slots", []):
                ff.write(json.dumps({
                    "chunk_id":    chunk_id,
                    "slot_index":  failed_slot.get("slot_index"),
                    "concept":     failed_slot.get("concept"),
                    "bloom_level": failed_slot.get("bloom_level"),
                    "reason":      failed_slot.get("reason"),
                }, ensure_ascii=False) + "\n")
                n_failed += 1

    return n_success, n_failed, n_sft


# Build chunk lookup for SFT record builder
chunks_lookup = {c.get("chunk_id", ""): c for c in s3_chunks}

n_success, n_failed, n_sft = split_results(
    all_results, S4_SUCCESS_PATH, S4_FAILED_PATH, S4_SFT_PATH, chunks_lookup
)

print(f"Success chunks (>=1 MCQ)   : {n_success}  -> {S4_SUCCESS_PATH}")
print(f"Failed slot records        : {n_failed}  -> {S4_FAILED_PATH}")
print(f"SFT training records       : {n_sft}   -> {S4_SFT_PATH}")

Success chunks (>=1 MCQ)   : 900  -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_success.jsonl
Failed slot records        : 4716  -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_failed.jsonl
SFT training records       : 900   -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_sft_dataset.jsonl


In [ ]:
import json
from typing import Dict, List

# If Drive is not mounted yet, run this first:
# from google.colab import drive
# drive.mount('/gdrive')

# Load S3 chunks
s3_chunks: List[Dict] = []
with open(S3_OUTPUT_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            s3_chunks.append(json.loads(line))

chunks_lookup = {c.get("chunk_id", ""): c for c in s3_chunks}

print(f"Loaded {len(s3_chunks)} S3 chunks.")
print(f"Chunk lookup size: {len(chunks_lookup)}")

# Load success records
success_results: List[Dict] = []
with open(S4_SUCCESS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            success_results.append(json.loads(line))

print(f"Loaded {len(success_results)} success records.")

# Build SFT file
written = 0
with open(S4_SFT_PATH, "w", encoding="utf-8") as f:
    for rec in success_results:
        source_chunk = chunks_lookup.get(rec["chunk_id"])
        if source_chunk is None:
            print("Missing chunk:", rec["chunk_id"])
            continue

        sft_rec = build_sft_record(source_chunk, rec)
        f.write(json.dumps(sft_rec, ensure_ascii=False) + "\n")
        written += 1

print(f"SFT records written: {written}")
print(f"Saved to: {S4_SFT_PATH}")

Loaded 555 S3 chunks.
Chunk lookup size: 555
Loaded 900 success records.
SFT records written: 900
Saved to: /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_sft_dataset.jsonl


In [ ]:
import json
import os
from typing import Dict, List, Tuple

# Paths
slot_path = S4_SLOTS_PATH
success_path = S4_SUCCESS_PATH
failed_path = S4_FAILED_PATH
sft_path = S4_SFT_PATH

# Load merged slot results
slot_records: List[Dict] = []
with open(slot_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            slot_records.append(json.loads(line))

print(f"Loaded {len(slot_records)} slot records.")

# Deduplicate by (chunk_id, slot_index)
# Keep the latest success if duplicate records exist
best_by_slot: Dict[Tuple[str, int], Dict] = {}

for rec in slot_records:
    key = (rec.get("chunk_id", ""), rec.get("slot_index", -1))
    if key not in best_by_slot:
        best_by_slot[key] = rec
    else:
        prev = best_by_slot[key]
        # Prefer success over failed
        if prev.get("status") != "success" and rec.get("status") == "success":
            best_by_slot[key] = rec

deduped_records = list(best_by_slot.values())
print(f"Deduplicated to {len(deduped_records)} unique slot records.")

# Group by chunk_id
chunks: Dict[str, Dict[str, List]] = {}
for rec in deduped_records:
    chunk_id = rec.get("chunk_id", "")
    chunks.setdefault(chunk_id, {"mcqs": [], "failed_slots": [], "meta": []})

# Build lookup from original S3 chunks
chunks_lookup = {c.get("chunk_id", ""): c for c in s3_chunks}

# Build success and failed outputs
success_results: List[Dict] = []
failed_records: List[Dict] = []

for chunk_id in chunks.keys():
    chunk_recs = [r for r in deduped_records if r.get("chunk_id", "") == chunk_id]

    success_mcqs = []
    failed_slots = []

    for rec in chunk_recs:
        if rec.get("status") == "success" and rec.get("mcq") is not None:
            success_mcqs.append(rec["mcq"])
        elif rec.get("status") == "failed":
            failed_slots.append({
                "chunk_id": chunk_id,
                "slot_index": rec.get("slot_index"),
                "concept": rec.get("concept"),
                "bloom_level": rec.get("bloom_level"),
                "reason": rec.get("reason"),
            })

    if success_mcqs:
        source_chunk = chunks_lookup.get(chunk_id, {})
        meta = source_chunk.get("metadata", {})
        success_results.append({
            "chunk_id": chunk_id,
            "bloom_level": meta.get("bloom_level", "understand"),
            "chunk_type": meta.get("chunk_type", "definition"),
            "mcqs": success_mcqs,
            "failed_slots": failed_slots,
            "total_slots": len(chunk_recs),
            "success_count": len(success_mcqs),
            "failure_count": len(failed_slots),
        })

    failed_records.extend(failed_slots)

# Write success file
with open(success_path, "w", encoding="utf-8") as f:
    for rec in success_results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Write failed file
with open(failed_path, "w", encoding="utf-8") as f:
    for rec in failed_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Build SFT file
with open(sft_path, "w", encoding="utf-8") as f:
    n_sft = 0
    for rec in success_results:
        source_chunk = chunks_lookup.get(rec["chunk_id"])
        if source_chunk:
            sft_rec = build_sft_record(source_chunk, rec)
            f.write(json.dumps(sft_rec, ensure_ascii=False) + "\n")
            n_sft += 1

print(f"Success chunks : {len(success_results)} -> {success_path}")
print(f"Failed slots   : {len(failed_records)} -> {failed_path}")
print(f"SFT records    : {n_sft} -> {sft_path}")

Loaded 6982 slot records.
Deduplicated to 2200 unique slot records.
Success chunks : 555 -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_success.jsonl
Failed slots   : 0 -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_failed.jsonl
SFT records    : 555 -> /gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_sft_dataset.jsonl


In [ ]:
print("S3 chunks loaded:", len(s3_chunks))
print("Chunk lookup size:", len(chunks_lookup))

matched = sum(1 for r in success_results if r["chunk_id"] in chunks_lookup)
print("Matched success records:", matched, "out of", len(success_results))

S3 chunks loaded: 555
Chunk lookup size: 555
Matched success records: 555 out of 555


In [ ]:
for r in success_results[:5]:
    print(r["chunk_id"], "->", r["chunk_id"] in chunks_lookup)

hands-on-machine-learning_672 -> True
hands-on-machine-learning_1250 -> True
andrew-ng-machine-learning-yearning_103 -> True
andrew-ng-machine-learning-yearning_7 -> True
hands-on-machine-learning_1054 -> True


In [ ]:
import json
from os.path import join
from typing import Dict, List

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
data_dir = "/gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen"

# Original chunk source (contains chunk text)
s3_input_path = join(data_dir, "selected_subset_v4_new.jsonl")

# Chunk-level validated MCQs
s4_input_path = join(data_dir, "s4_success.jsonl")

# Final slot-level clean file
output_path = join(data_dir, "s4_success_per_slot.jsonl")

# ------------------------------------------------------------------
# Load S3 chunks and build chunk_id -> chunk_text mapping
# ------------------------------------------------------------------
chunk_text_map: Dict[str, str] = {}

with open(s3_input_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        rec = json.loads(line)

        chunk_id = rec.get("chunk_id")

        # Adjust this field name if your text field is different
        chunk_text = rec.get("text", "")

        if chunk_id:
            chunk_text_map[chunk_id] = chunk_text

print(f"Loaded {len(chunk_text_map)} chunk texts from S3.")

# ------------------------------------------------------------------
# Load chunk-level success file
# ------------------------------------------------------------------
chunk_records: List[Dict] = []

with open(s4_input_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if line:
            chunk_records.append(json.loads(line))

print(f"Loaded {len(chunk_records)} chunk-level MCQ records.")

# ------------------------------------------------------------------
# Convert chunk-level -> slot-level
# Add chunk text from S3 mapping
# ------------------------------------------------------------------
slot_records: List[Dict] = []

for chunk_rec in chunk_records:

    chunk_id    = chunk_rec["chunk_id"]
    chunk_type  = chunk_rec["chunk_type"]
    mcqs        = chunk_rec["mcqs"]

    # Retrieve original chunk text
    chunk_text = chunk_text_map.get(chunk_id, "")

    for mcq in mcqs:

        slot_record = {
            "chunk_id": chunk_id,
            "text": chunk_text,
            "slot_index": mcq.get("slot_index"),
            "chunk_type": chunk_type,
            "concept": mcq.get("concept"),
            "bloom_level": mcq.get("bloom_level"),

            "mcq": {
                "question": mcq.get("question"),
                "options": mcq.get("options"),
                "answer": mcq.get("answer"),
                "explanation": mcq.get("explanation")
            }
        }

        slot_records.append(slot_record)

# ------------------------------------------------------------------
# Save final slot-level validation dataset
# ------------------------------------------------------------------
with open(output_path, "w", encoding="utf-8") as f:
    for rec in slot_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\nSaved {len(slot_records)} slot-level records.")
print(f"Output file:\n{output_path}")

Loaded 555 chunk texts from S3.
Loaded 555 chunk-level MCQ records.

Saved 2200 slot-level records.
Output file:
/gdrive/MyDrive/GP_Quiz-Generation_Data_New_Gen/s4_success_per_slot.jsonl


## 11. Output Inspection

In [ ]:
success_results: List[Dict] = []
with open(S4_SUCCESS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            success_results.append(json.loads(line))

all_mcqs = [mcq for r in success_results for mcq in r.get("mcqs", [])]
print(f"Success chunks : {len(success_results)}")
print(f"Total MCQs     : {len(all_mcqs)}")
print(f"Avg MCQs/chunk : {len(all_mcqs) / max(1, len(success_results)):.2f}")
print(f"  (expected: {(3+4)/2:.1f} avg based on dynamic slot count)\n")

if success_results:
    sample = random.choice(success_results)
    print(f"=== {sample['chunk_id']}  [{sample['bloom_level']} | {sample['chunk_type']}] ===")
    print(f"    Slots: {sample['total_slots']} total | "
          f"{sample['success_count']} success | {sample['failure_count']} failed\n")
    for i, mcq in enumerate(sample["mcqs"], 1):
        slot_tag = f"[slot {mcq.get('slot_index',i-1)+1} . {mcq.get('slot_type','named')}]"
        print(f"--- MCQ {i} {slot_tag} | concept: {mcq['concept']} ---")
        print(f"Q: {mcq['question']}")
        for key, val in mcq["options"].items():
            marker = "+" if key == mcq["answer"] else " "
            print(f"  {marker} {key}) {val}")
        print(f"Explanation: {mcq['explanation']}\n")

Success chunks : 555
Total MCQs     : 2200
Avg MCQs/chunk : 3.96
  (expected: 3.5 avg based on dynamic slot count)

=== hands-on-machine-learning_1084  [analyze | comparison] ===
    Slots: 4 total | 4 success | 0 failed

--- MCQ 1 [slot 1 . named] | concept: undercomplete autoencoder ---
Q: In what way is an undercomplete autoencoder distinct from a model that can trivially copy inputs?
    A) It can easily replicate the input data directly into its codings without feature extraction.
    B) It is characterized by an internal representation with a higher dimensionality than the input data.
    C) It is designed with a higher dimensionality in its internal representation compared to the input data.
  + D) It is compelled to learn and retain only the most significant features from the input data.
Explanation: An undercomplete autoencoder has a lower-dimensional internal representation than the input data, forcing it to learn the most important features and discard less important ones to

## 12. Quality Statistics

In [ ]:
bloom_dist  = Counter(mcq.get("bloom_level", "?") for mcq in all_mcqs)
answer_dist = Counter(mcq.get("answer",      "?") for mcq in all_mcqs)
type_dist   = Counter(mcq.get("slot_type",   "?") for mcq in all_mcqs)

print("-- Bloom Level Distribution -----------------------------------------")
for level in ["remember","understand","apply","analyze","evaluate","create"]:
    bar = "#" * bloom_dist.get(level, 0)
    print(f"  {level:<12}: {bloom_dist.get(level, 0):>5}  {bar}")

print("\n-- Answer Key Distribution (should be balanced) ---------------------")
for key in ["A","B","C","D"]:
    bar = "#" * (answer_dist.get(key, 0) // max(1, len(all_mcqs) // 40))
    print(f"  {key}: {answer_dist.get(key, 0):>5}  {bar}")

if all_mcqs:
    max_share = max(answer_dist.values()) / len(all_mcqs)
    print(f"\n  Max position share: {max_share:.1%}  "
          f"{'OK balanced' if max_share < 0.35 else 'WARNING biased -- check shuffle'}")

print("\n-- Slot Type Distribution --------------------------------------------")
for stype in ["named","reuse","inferred"]:
    print(f"  {stype:<10}: {type_dist.get(stype, 0):>5}")

# Failed slot analysis
failed_records: List[Dict] = []
if os.path.exists(S4_FAILED_PATH):
    with open(S4_FAILED_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                failed_records.append(json.loads(line))

print(f"\n-- Failure Summary --------------------------------------------------")
print(f"  Total failed slots   : {len(failed_records)}")
if failed_records:
    fail_reasons = Counter(r.get("reason","?")[:50] for r in failed_records)
    for reason, count in fail_reasons.most_common(5):
        print(f"    {count:>4}x  {reason}")

# Stage file stats (v5)
print(f"\n-- Stage File Stats (v5) --------------------------------------------")
for label, path in [
    ("Stage 1 raw records  ", S4_RAW_PATH),
    ("Stage 2 parsed records", S4_PARSED_PATH),
    ("Stage 3 slot results ", S4_SLOTS_PATH),
]:
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            n = sum(1 for line in f if line.strip())
        print(f"  {label}: {n:>6} records  -> {path}")
    else:
        print(f"  {label}: (not found) -> {path}")

# Slot result breakdown from Stage 3
if os.path.exists(S4_SLOTS_PATH):
    slot_records: List[Dict] = []
    with open(S4_SLOTS_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                slot_records.append(json.loads(line))
    slot_status = Counter(r.get("status", "?") for r in slot_records)
    print(f"\n  Slot checkpoint breakdown:")
    for status, count in slot_status.items():
        print(f"    {status:<10}: {count:>5}")


-- Bloom Level Distribution -----------------------------------------
  remember    :    43  ###########################################
  understand  :   500  ####################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################
  apply       :   668  ############################################################################################################################################################################################################################################################################################################################